In [6]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import time

In [16]:
# GHOST EFFECT (improved)

import cv2
import numpy as np
import time
from collections import deque

capture = cv2.VideoCapture(0)
time.sleep(2)

# for bg
frames = []
for i in range(60):
    return_val, frame= capture.read()
    if return_val:
        frames.append(cv2.flip(frame, 1))
background= np.median(frames, axis=0).astype(np.uint8)
background_blur= cv2.GaussianBlur(background, (5, 5), 0)

kernel= np.ones((5, 5), np.uint8)

DELAY= 30                          # bigger = ghost further behind 
past_frames= deque(maxlen=DELAY)
past_masks= deque(maxlen=DELAY)


def add_ghost(output, old_frame, old_mask, current_mask, strength):
    # ghost only where you WERE, but are NOT now
    ghost_area= cv2.bitwise_and(old_mask, cv2.bitwise_not(current_mask))

    # soft edges: blur the mask so the ghost blends in
    soft= cv2.GaussianBlur(ghost_area, (21, 21), 0)
    alpha= (soft / 255.0) * strength
    alpha[current_mask > 0]= 0          # keep the real person fully solid
    alpha= alpha[:, :, np.newaxis]      # make it work on 3 colour channels

    # mix old frame into output using alpha
    output= output * (1 - alpha) + old_frame * alpha
    return output.astype(np.uint8)


while capture.isOpened():
    return_val, img= capture.read()
    if not return_val:
        break
    img= cv2.flip(img, 1)

    # ---- Background subtraction: find the person ----
    img_blur= cv2.GaussianBlur(img, (5, 5), 0)
    diff= cv2.absdiff(img_blur, background_blur)
    gray= cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, mask= cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)

    # ---- Clean the mask ----
    mask= cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask= cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # keep only big shapes (your body), remove small noise blobs
    contours, _= cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean= np.zeros_like(mask)
    for c in contours:
        if cv2.contourArea(c) > 3000:
            cv2.drawContours(clean, [c], -1, 255, -1)   # -1 = fill inside
    mask= cv2.dilate(clean, kernel, iterations=1)

    output= img.copy()

    # ---- Two ghosts: far one first, then near one on top ----
    if len(past_frames) == DELAY:
        output= add_ghost(output, past_frames[0], past_masks[0], mask, 0.5)           
        

    past_frames.append(img.copy())
    past_masks.append(mask.copy())

    cv2.imshow("Ghost Effect", output)
    cv2.imshow("Mask", mask)

    k= cv2.waitKey(10)
    if k == 27:   # ESC to quit
        break

capture.release()
cv2.destroyAllWindows()

In [11]:

# INVISIBILITY EFFECT

import cv2
import numpy as np
import time

capture = cv2.VideoCapture(0)
time.sleep(2)

# ---- Capture background: take median of 60 frames ----
frames= []
for i in range(60):
    return_val, frame= capture.read()
    if return_val:
        frames.append(cv2.flip(frame, 1))
background= np.median(frames, axis=0).astype(np.uint8)

kernel= np.ones((5, 5), np.uint8)

while capture.isOpened():
    return_val, img= capture.read()
    if not return_val:
        break
    img= cv2.flip(img, 1)

    hsv= cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Red range 1 (hue 0-10)
    lower_red= np.array([0, 40, 40])
    upper_red= np.array([10, 255, 255])
    mask1= cv2.inRange(hsv, lower_red, upper_red)

    # Red range 2 (your original range)
    lower_red= np.array([155, 40, 40])
    upper_red= np.array([180, 255, 255])
    mask2= cv2.inRange(hsv, lower_red, upper_red)

    mask1= cv2.bitwise_or(mask1, mask2)

    # Clean the mask
    mask1= cv2.morphologyEx(mask1, cv2.MORPH_OPEN, kernel, iterations=2)   # remove noise
    mask1= cv2.morphologyEx(mask1, cv2.MORPH_CLOSE, kernel, iterations=2)  # fill holes
    mask1= cv2.dilate(mask1, kernel, iterations=1)                         # cover edges
    mask1= cv2.medianBlur(mask1, 5)                                        # smooth edges

    mask2= cv2.bitwise_not(mask1)

    res1= cv2.bitwise_and(background, background, mask=mask1)  # background where cloth is
    res2= cv2.bitwise_and(img, img, mask=mask2)                # current frame everywhere else

    final_output= cv2.addWeighted(res1, 1, res2, 1, 0)

    cv2.imshow("Invisibility Effect", final_output)
    

    k= cv2.waitKey(10)
    if k == 27:   # ESC to quit
        break

capture.release()
cv2.destroyAllWindows()